# B2.9 · Idempotency, replay and rollback

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

---

**Risk.** Any agent action you cannot replay you cannot investigate.

**Control.** Design requirements, not afterthoughts.

**This lab.** Replay an agent run from its trace.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.9"))

Idempotency, replay and rollback. The agent will do the same thing twice; the only question is whether that is harmless.

In [ ]:
class Ledger:
    """Idempotency keys: the same operation, applied twice, lands once."""
    def __init__(self): self.applied, self.log = {}, []
    def apply(self, key, op, amount):
        if key in self.applied:
            self.log.append(f"skip  {key} ({op}) — already applied")
            return False
        self.applied[key] = (op, amount)
        self.log.append(f"apply {key} ({op} {amount})")
        return True

led = Ledger()
for key, op, amt in [("pr-42-merge", "merge", 1), ("pr-42-merge", "merge", 1),
                     ("pr-43-merge", "merge", 1)]:
    led.apply(key, op, amt)
print("\n".join(led.log))
print("\napplied operations:", len(led.applied), "(three calls, two effects)")

Now replay — the forensics side of the same property.

In [ ]:
from cybercommons import ir

for name, r in (("fully instrumented", ir.Replay(["p1"], ["tool result"], "glm-4.6", 0)),
                ("typical production",  ir.Replay(["p1"], [], "", None))):
    ok, missing = r.replayable()
    print(f"{name:20s} replayable={ok}")
    for m in missing:
        print(f"    ✗ {m}")

### Expect

Three apply calls produce two effects — the duplicate is skipped. The instrumented run is replayable; the typical one is missing tool results, model version and seed.

### Your turn

Which of your tools are naturally idempotent, and which need a key? `post_comment` is the one people get wrong — a duplicated comment is noise, but a duplicated *approval* is not.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.9.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*